In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/amazonml2025-absolut/train.csv
/kaggle/input/amazonml2025-absolut/test.csv


In [2]:
## 1
import pandas as pd

In [3]:
#2
traind = pd.read_csv('/kaggle/input/amazonml2025-absolut/train.csv')
traind.head()

,sample_id,catalog_content,image_link,price
0,33127,"Item Name: La Victoria Green Taco Sauce Mild, ...",https://m.media-amazon.com/images/I/51mo8htwTH...,4.89
1,198967,"Item Name: Salerno Cookies, The Original Butte...",https://m.media-amazon.com/images/I/71YtriIHAA...,13.12
2,261251,"Item Name: Bear Creek Hearty Soup Bowl, Creamy...",https://m.media-amazon.com/images/I/51+PFEe-w-...,1.97
3,55858,Item Name: Judee’s Blue Cheese Powder 11.25 oz...,https://m.media-amazon.com/images/I/41mu0HAToD...,30.34
4,292686,"Item Name: kedem Sherry Cooking Wine, 12.7 Oun...",https://m.media-amazon.com/images/I/41sA037+Qv...,66.49


In [4]:
#3
import pandas as pd
import re

def extract_value_unit_packs(text):
    # Extract value
    value_match = re.search(r'Value:\s*([\d.]+)', text)
    value = float(value_match.group(1)) if value_match else None

    # Extract unit
    unit_match = re.search(r'Unit:\s*(\w+)', text)
    unit = unit_match.group(1) if unit_match else None

    # Extract number of packs
    pack_match = re.search(r'Pack of\s*(\d+)', text, re.IGNORECASE)
    num_of_packs = int(pack_match.group(1)) if pack_match else 1

    return pd.Series([value, unit, num_of_packs])
traind[['value_extracted', 'unit_extracted', 'num_of_packs']] = traind['catalog_content'].apply(extract_value_unit_packs)
# Convert 'catalog_content' to lowercase
traind['catalog_content'] = traind['catalog_content'].str.lower()

# Convert 'unit_extracted' to lowercase (only if it's a string)
traind['unit_extracted'] = traind['unit_extracted'].apply(lambda x: x.lower() if isinstance(x, str) else x)
# Filtered copy
traind_filtered = traind[
    (traind['value_extracted'].notnull()) &
    (traind['value_extracted'] != 0) &
    (traind['unit_extracted'].notnull())
].copy()
print(traind_filtered.shape)

(74018, 7)


In [5]:
#4
traind_filtered.head()

,sample_id,catalog_content,image_link,price,value_extracted,unit_extracted,num_of_packs
0,33127,"item name: la victoria green taco sauce mild, ...",https://m.media-amazon.com/images/I/51mo8htwTH...,4.89,72.00,fl,6.0
1,198967,"item name: salerno cookies, the original butte...",https://m.media-amazon.com/images/I/71YtriIHAA...,13.12,32.00,ounce,4.0
2,261251,"item name: bear creek hearty soup bowl, creamy...",https://m.media-amazon.com/images/I/51+PFEe-w-...,1.97,11.40,ounce,6.0
3,55858,item name: judee’s blue cheese powder 11.25 oz...,https://m.media-amazon.com/images/I/41mu0HAToD...,30.34,11.25,ounce,1.0
4,292686,"item name: kedem sherry cooking wine, 12.7 oun...",https://m.media-amazon.com/images/I/41sA037+Qv...,66.49,12.00,count,1.0


In [6]:
#5

# =========================================================
# 📦 Imports
# =========================================================
import os
import pandas as pd
import numpy as np
import torch
import torchvision.transforms as transforms
from torchvision import models
from PIL import Image
from io import BytesIO
import requests
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from torch import nn
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

torch.backends.cudnn.benchmark = True  # optimize CNN on GPU

# =========================================================
# 📁 Dataset Preprocessing
# =========================================================
df = traind_filtered.copy()
# df = df.iloc[:100:, ::]
df['catalog_content'] = df['catalog_content'].str.lower()

# --- Tabular Features ---
unit_encoder = OneHotEncoder(handle_unknown='ignore')
unit_encoded = unit_encoder.fit_transform(df[['unit_extracted']].fillna('unknown')).toarray()

tabular_scaler = StandardScaler()
scaled_features = tabular_scaler.fit_transform(df[['value_extracted', 'num_of_packs']].fillna(0))
tabular_data = np.hstack([scaled_features, unit_encoded])

# --- Text Features ---
tfidf = TfidfVectorizer(max_features=100)
text_data = tfidf.fit_transform(df['catalog_content'].fillna("")).toarray()

# --- Targets ---
targets = df['price'].values

# =========================================================
# 🖼️ Image Preprocessing
# =========================================================
transform = transforms.Compose([
    transforms.Resize((300, 300)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

def fetch_image_from_url(url):
    try:
        response = requests.get(url, timeout=5)
        img = Image.open(BytesIO(response.content)).convert('RGB')
        return img
    except:
        return Image.new('RGB', (224, 224), color=(255, 255, 255))

# =========================================================
# 🧮 Dataset Class
# =========================================================
class PriceDataset(Dataset):
    def __init__(self, df, tabular_data, text_data, targets=None, transform=None):
        self.df = df
        self.tabular_data = tabular_data
        self.text_data = text_data
        self.targets = targets
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = fetch_image_from_url(row['image_link'])
        if self.transform:
            image = self.transform(image)

        tabular = torch.tensor(self.tabular_data[idx], dtype=torch.float32)
        text = torch.tensor(self.text_data[idx], dtype=torch.float32)

        if self.targets is not None:
            target = torch.tensor(self.targets[idx], dtype=torch.float32)
            return image, tabular, text, target
        else:
            return image, tabular, text

# =========================================================
# 🧠 Model Architecture
# =========================================================
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights

class PricePredictor(nn.Module):
    def __init__(self, tabular_dim, text_dim):
        super().__init__()
        self.cnn = efficientnet_b3(weights=EfficientNet_B3_Weights.DEFAULT)
        self.cnn.classifier = nn.Identity()

        for name, param in self.cnn.named_parameters():
            if any(f"features.{i}" in name for i in range(7)):
                param.requires_grad = False

        self.tabular_fc = nn.Sequential(
            nn.Linear(tabular_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

        self.text_fc = nn.Sequential(
            nn.Linear(text_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2)
        )

        self.final_fc = nn.Sequential(
            nn.Linear(1536 + 64 + 64, 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )

    def forward(self, image, tabular, text):
        img_feat = self.cnn(image)
        tab_feat = self.tabular_fc(tabular)
        text_feat = self.text_fc(text)
        combined = torch.cat([img_feat, tab_feat, text_feat], dim=1)
        return self.final_fc(combined).squeeze()

# =========================================================
# 📐 SMAPE Loss Function
# =========================================================
def smape_loss(y_pred, y_true, epsilon=1e-8):
    numerator = torch.abs(y_pred - y_true)
    denominator = (torch.abs(y_true) + torch.abs(y_pred)).clamp(min=epsilon) / 2
    return torch.mean(numerator / denominator)

# =========================================================
# 🏋️ Training Setup
# =========================================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Training on: {device}")

model = PricePredictor(tabular_dim=tabular_data.shape[1], text_dim=text_data.shape[1]).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
amp_scaler = torch.cuda.amp.GradScaler()

# =========================================================
# 💾 Checkpoint Setup
# =========================================================
checkpoint_path = "price_predictor_checkpoint-2.pth"
start_epoch = 0

if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    amp_scaler.load_state_dict(checkpoint['scaler_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    print(f"🔁 Resuming from epoch {start_epoch}")

# =========================================================
# 📦 DataLoader
# =========================================================
train_dataset = PriceDataset(df, tabular_data, text_data, targets, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, pin_memory=True)

# =========================================================
# 🔁 Training Loop with Checkpointing
# =========================================================
num_epochs = 4

for epoch in range(start_epoch, num_epochs):
    model.train()
    total_loss = 0
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=True)

    for image, tabular, text, target in loop:
        image = image.to(device, non_blocking=True)
        tabular = tabular.to(device, non_blocking=True)
        text = text.to(device, non_blocking=True)
        target = target.to(device, non_blocking=True)

        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            output = model(image, tabular, text)
            loss = smape_loss(output, target)

        amp_scaler.scale(loss).backward()
        amp_scaler.step(optimizer)
        amp_scaler.update()

        total_loss += loss.item()
        loop.set_postfix(SMAPE=loss.item())

    print(f"✅ Epoch {epoch+1} completed | Avg SMAPE: {total_loss / len(train_loader):.4f}")

    # Save checkpoint
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scaler_state_dict': amp_scaler.state_dict()
    }, checkpoint_path)
    print(f"💾 Saved checkpoint at epoch {epoch+1}")

print("🎉 Training complete!")


✅ Training on: cuda


Downloading: "https://download.pytorch.org/models/efficientnet_b3_rwightman-b3899882.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b3_rwightman-b3899882.pth
100%|██████████| 47.2M/47.2M [00:00<00:00, 186MB/s]
/tmp/ipykernel_37/2104514979.py:148: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  amp_scaler = torch.cuda.amp.GradScaler()


🔁 Resuming from epoch 3


Epoch 4:   0%|          | 0/1157 [00:00<?, ?it/s]/tmp/ipykernel_37/2104514979.py:187: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 4: 100%|██████████| 1157/1157 [3:43:39<00:00, 11.60s/it, SMAPE=0.494] 


✅ Epoch 4 completed | Avg SMAPE: 0.5576
💾 Saved checkpoint at epoch 4
🎉 Training complete!


In [7]:
print(model)

PricePredictor(
  (cnn): EfficientNet(
    (features): Sequential(
      (0): Conv2dNormActivation(
        (0): Conv2d(3, 40, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (1): BatchNorm2d(40, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): SiLU(inplace=True)
      )
      (1): Sequential(
        (0): MBConv(
          (block): Sequential(
            (0): Conv2dNormActivation(
              (0): Conv2d(40, 40, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=40, bias=False)
              (1): BatchNorm2d(40, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
              (2): SiLU(inplace=True)
            )
            (1): SqueezeExcitation(
              (avgpool): AdaptiveAvgPool2d(output_size=1)
              (fc1): Conv2d(40, 10, kernel_size=(1, 1), stride=(1, 1))
              (fc2): Conv2d(10, 40, kernel_size=(1, 1), stride=(1, 1))
              (activation): SiLU(inplace=True)
              (scale_

## Predictions

In [8]:
#6
import joblib

joblib.dump(unit_encoder, "unit_encoder.pkl")
joblib.dump(tabular_scaler, "tabular_scaler.pkl")
joblib.dump(tfidf, "tfidf_vectorizer.pkl")

['tfidf_vectorizer.pkl']

In [10]:
#7

import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import re
# =========================================================
# 📁 Load Test Data
# =========================================================
test_df = pd.read_csv('/kaggle/input/amazonml2025-absolut/test.csv')
##change accordingly
#test_df = test_df.iloc[:200, :]  # for demo/testing
test_df['catalog_content'] = test_df['catalog_content'].str.lower()

# =========================================================
# 🧪 Feature Extraction Function
# =========================================================
def extract_value_unit_packs(text):
    value_match = re.search(r'Value:\s*([\d.]+)', text)
    value = float(value_match.group(1)) if value_match else 0.0

    unit_match = re.search(r'Unit:\s*(\w+)', text)
    unit = unit_match.group(1).lower() if unit_match else 'unknown'

    pack_match = re.search(r'Pack of\s*(\d+)', text, re.IGNORECASE)
    num_of_packs = int(pack_match.group(1)) if pack_match else 1

    return pd.Series([value, unit, num_of_packs])

# =========================================================
# 🧮 Test Dataset Class
# =========================================================
class TestDataset(Dataset):
    def __init__(self, df, transform, tabular_data, text_data):
        self.df = df
        self.transform = transform
        self.tabular_data = tabular_data
        self.text_data = text_data

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = fetch_image_from_url(row['image_link'])
        image = self.transform(image)

        tabular = torch.tensor(self.tabular_data[idx], dtype=torch.float32)
        text = torch.tensor(self.text_data[idx], dtype=torch.float32)

        return image, tabular, text

# =========================================================
# 🔄 Chunked Prediction and Saving
# =========================================================
##change accordingly
chunk_size = 10000
num_chunks = len(test_df) // chunk_size + int(len(test_df) % chunk_size != 0)

print(f"🔹 Starting prediction on full dataset with {len(test_df)} rows ({num_chunks} chunks)")
for i in range(num_chunks):
    print(f"🔄 Processing chunk {i+1}/{num_chunks}")

    # Slice the chunk
    chunk_df = test_df.iloc[i * chunk_size : (i + 1) * chunk_size].copy()
    chunk_df['catalog_content'] = chunk_df['catalog_content'].str.lower()

    # Extract tabular features
    chunk_df[['value_extracted', 'unit_extracted', 'num_of_packs']] = chunk_df['catalog_content'].apply(extract_value_unit_packs)
    unit_encoded_chunk = unit_encoder.transform(chunk_df[['unit_extracted']]).toarray()
    scaled_features_chunk = tabular_scaler.transform(chunk_df[['value_extracted', 'num_of_packs']])
    tabular_chunk = np.hstack([scaled_features_chunk, unit_encoded_chunk])

    # Text features
    text_chunk = tfidf.transform(chunk_df['catalog_content'].fillna("")).toarray()

    # Dataset and DataLoader
    chunk_dataset = TestDataset(chunk_df, transform, tabular_chunk, text_chunk)
    chunk_loader = DataLoader(chunk_dataset, batch_size=32, shuffle=False, pin_memory=True)

    # Predictions
    model.eval()
    chunk_predictions = []

    with torch.no_grad():
        for image, tabular, text in tqdm(chunk_loader, desc=f"Predicting Chunk {i+1}"):
            image = image.to(device, non_blocking=True)
            tabular = tabular.to(device, non_blocking=True)
            text = text.to(device, non_blocking=True)

            with torch.cuda.amp.autocast():
                output = model(image, tabular, text)

            chunk_predictions.extend(output.cpu().numpy())

    # Save chunk submission
    submission_chunk = pd.DataFrame({
        'sample_id': chunk_df['sample_id'],
        'price': chunk_predictions
    })
    filename = f"submission_epoch4_chunk{i+1}_rows{len(chunk_df)}.csv"
    submission_chunk.to_csv(filename, index=False)
    print(f"✅ Saved {filename}")


🔹 Starting prediction on full dataset with 75000 rows (8 chunks)
🔄 Processing chunk 1/8


Predicting Chunk 1:   0%|          | 0/313 [00:00<?, ?it/s]/tmp/ipykernel_37/3759802888.py:93: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Predicting Chunk 1: 100%|██████████| 313/313 [31:22<00:00,  6.01s/it]


✅ Saved submission_epoch4_chunk1_rows10000.csv
🔄 Processing chunk 2/8


Predicting Chunk 2:   0%|          | 0/313 [00:00<?, ?it/s]/tmp/ipykernel_37/3759802888.py:93: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Predicting Chunk 2: 100%|██████████| 313/313 [31:43<00:00,  6.08s/it]


✅ Saved submission_epoch4_chunk2_rows10000.csv
🔄 Processing chunk 3/8


...

✅ Saved submission_epoch4_chunk6_rows10000.csv
🔄 Processing chunk 7/8


Predicting Chunk 7:   0%|          | 0/313 [00:00<?, ?it/s]/tmp/ipykernel_37/3759802888.py:93: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Predicting Chunk 7: 100%|██████████| 313/313 [31:16<00:00,  6.00s/it]


✅ Saved submission_epoch4_chunk7_rows10000.csv
🔄 Processing chunk 8/8


Predicting Chunk 8:   0%|          | 0/157 [00:00<?, ?it/s]/tmp/ipykernel_37/3759802888.py:93: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Predicting Chunk 8: 100%|██████████| 157/157 [14:33<00:00,  5.56s/it]

✅ Saved submission_epoch4_chunk8_rows5000.csv


# the session has restarted because of time limit

In [ ]:
print(model)

In [11]:
torch.save(model, "new_price_predictor_full_model_epoch_4.pth")


In [12]:
import pickle

with open("unit_encoder_epoch_4.pkl", "wb") as f:
    pickle.dump(unit_encoder, f)

with open("tabular_scaler_epoch_4.pkl", "wb") as f:
    pickle.dump(tabular_scaler, f)

with open("tfidf_epoch_4.pkl", "wb") as f:
    pickle.dump(tfidf, f)